<a href="https://colab.research.google.com/github/Manarsenic/EDA/blob/main/EDA_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from scipy.stats import zscore

# A dictionary of URLs for all the datasets you provided.
urls = {'undernourishment': "https://ourworldindata.org/grapher/prevalence-of-undernourishment.csv",
    'calorie_supply': "https://ourworldindata.org/grapher/daily-per-capita-caloric-supply.csv",
    'agri_employment': "https://ourworldindata.org/grapher/share-of-the-labor-force-employed-in-agriculture.csv",
    'stunting_rate': "https://ourworldindata.org/grapher/share-of-children-younger-than-5-who-suffer-from-stunting.csv",
    'severe_food_insecurity': "https://ourworldindata.org/grapher/share-of-population-with-severe-food-insecurity.csv",
    'global_hunger_index': "https://ourworldindata.org/grapher/global-hunger-index.csv",
    'children_wasted': "https://ourworldindata.org/grapher/share-of-children-with-a-weight-too-low-for-their-height-wasting.csv",
    'food_loss_index': "https://ourworldindata.org/grapher/global-food-loss-index.csv",
    'price_of_healthy':"https://ourworldindata.org/grapher/cost-foods-healthy-diet.csv?v=1&csvType=full&useColumnShortNames=true "}

all_data = {}

# Scrape each dataset and store it.
for name, url in urls.items():
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        all_data[name] = pd.read_csv(StringIO(resp.text))
        print(f"Successfully scraped '{name}' data.")
    except requests.exceptions.RequestException as e:
        print(f"Error scraping '{name}': {e}")

# Merge all data into a single DataFrame.
df = all_data['undernourishment']
for name, df_to_merge in all_data.items():
    if name != 'undernourishment':
        df = pd.merge(df, df_to_merge, on=['Entity', 'Code', 'Year'], how='outer', suffixes=(f'_{name}', ''))

print("\nAll data scraped and merged into a single DataFrame.")
df.info()
print("\nMerged DataFrame Preview:\n", df.head())
df

Successfully scraped 'undernourishment' data.
Successfully scraped 'calorie_supply' data.
Successfully scraped 'agri_employment' data.
Successfully scraped 'stunting_rate' data.
Successfully scraped 'severe_food_insecurity' data.
Successfully scraped 'global_hunger_index' data.
Successfully scraped 'children_wasted' data.
Successfully scraped 'food_loss_index' data.
Successfully scraped 'price_of_healthy' data.

All data scraped and merged into a single DataFrame.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17026 entries, 0 to 17025
Data columns (total 18 columns):
 #   Column                                                                                                                           Non-Null Count  Dtype  
---  ------                                                                                                                           --------------  -----  
 0   Entity                                                                                               

,Entity,Code,Year,2.1.1 Prevalence of undernourishment | 000000000024000 || Value | 006121 || percent,Daily calorie supply per person,share_employed_agri,"Stunting prevalence among children under 5 years of age (% height-for-age <-2 SD), model-based estimates - Sex: both sexes",Prevalence of severe food insecurity in the total population (percent) (3-year average) | 00210401 || Value | 006121 || percent,Global Hunger Index (2021),411773-annotations,"Prevalence of wasting, weight for height (% of children under 5)",12.3.1 - Global food loss index - AG_FLS_INDEX,Cost of starchy staples,"Cost of legumes, nuts and seeds",Cost of oils and fats,Cost of fruits,Cost of vegetables,Cost of animal-source foods
0,Afghanistan,AFG,1961,NaN,2914.3633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,1962,NaN,2835.6210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,1963,NaN,2623.7139,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,1964,NaN,2872.5825,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,1965,NaN,2876.1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17021,Zimbabwe,ZWE,2020,39.5,2035.4940,NaN,23.5,31.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17022,Zimbabwe,ZWE,2021,38.9,2083.9954,NaN,23.3,28.6,27.5,Value represents the mid-point of its group in...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17023,Zimbabwe,ZWE,2022,38.1,2072.4504,NaN,23.5,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17024,Zimbabwe,ZWE,2023,NaN,NaN,NaN,23.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Assuming 'df' is your combined DataFrame from the previous step.

# Identify the columns that contain numerical data and need imputation.
# The 'country' column is categorical, so we exclude it.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# The 'Code' column is also not useful for imputation, so we remove it.
# We also want to impute all the key indicator columns.
impute_cols = [col for col in numeric_cols if col not in ['Code', 'Year']]

# Create an instance of the KNNImputer. We'll use 5 neighbors.
imputer = KNNImputer(n_neighbors=5)

# Fit and transform the data to fill in the missing values.
# The imputer returns a NumPy array.
imputed_data = imputer.fit_transform(df[impute_cols])

# Convert the imputed NumPy array back to a DataFrame.
df[impute_cols] = pd.DataFrame(imputed_data, columns=impute_cols)

# Display a preview to see how the missing values have been filled.
print("DataFrame after k-NN Imputation:\n")
df.info()
print(df.sample(5))

# You now have a complete dataset with no null values in the key indicator columns.
# This prepares your data for the EDA phase.
df

DataFrame after k-NN Imputation:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17026 entries, 0 to 17025
Data columns (total 18 columns):
 #   Column                                                                                                                           Non-Null Count  Dtype  
---  ------                                                                                                                           --------------  -----  
 0   Entity                                                                                                                           17026 non-null  object 
 1   Code                                                                                                                             11577 non-null  object 
 2   Year                                                                                                                             17026 non-null  int64  
 3   2.1.1 Prevalence of undernourishment | 000000000024000 || Valu

,Entity,Code,Year,2.1.1 Prevalence of undernourishment | 000000000024000 || Value | 006121 || percent,Daily calorie supply per person,share_employed_agri,"Stunting prevalence among children under 5 years of age (% height-for-age <-2 SD), model-based estimates - Sex: both sexes",Prevalence of severe food insecurity in the total population (percent) (3-year average) | 00210401 || Value | 006121 || percent,Global Hunger Index (2021),411773-annotations,"Prevalence of wasting, weight for height (% of children under 5)",12.3.1 - Global food loss index - AG_FLS_INDEX,Cost of starchy staples,"Cost of legumes, nuts and seeds",Cost of oils and fats,Cost of fruits,Cost of vegetables,Cost of animal-source foods
0,Afghanistan,AFG,1961,6.52,2914.36330,20.432000,16.76,11.54,11.76,NaN,6.356,99.386667,0.586,0.398,0.180,0.800,0.896,0.944
1,Afghanistan,AFG,1962,6.56,2835.62100,21.916000,18.86,7.76,13.24,NaN,4.200,99.386667,0.596,0.456,0.210,0.608,0.746,1.094
2,Afghanistan,AFG,1963,13.04,2623.71390,13.922000,14.22,26.58,19.68,NaN,6.160,99.386667,0.540,0.488,0.212,0.812,0.856,1.064
3,Afghanistan,AFG,1964,5.02,2872.58250,10.804000,13.92,5.42,14.08,NaN,3.920,99.386667,0.552,0.406,0.156,0.738,0.844,1.100
4,Afghanistan,AFG,1965,4.32,2876.19600,16.668000,16.04,3.42,9.08,NaN,5.880,99.386667,0.572,0.388,0.152,0.622,0.616,1.134
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17021,Zimbabwe,ZWE,2020,39.50,2035.49400,50.900001,23.50,31.30,42.52,NaN,5.880,99.186000,0.666,0.316,0.136,0.468,0.788,1.156
17022,Zimbabwe,ZWE,2021,38.90,2083.99540,43.746001,23.30,28.60,27.50,Value represents the mid-point of its group in...,7.180,99.186000,0.708,0.336,0.142,0.428,0.782,1.192
17023,Zimbabwe,ZWE,2022,38.10,2072.45040,48.540000,23.50,26.00,34.64,NaN,4.300,99.186000,0.666,0.316,0.136,0.468,0.788,1.156
17024,Zimbabwe,ZWE,2023,13.76,2667.11926,34.274346,23.50,16.62,23.92,NaN,7.222,98.344000,0.622,0.366,0.198,0.392,0.542,1.228
